# Path B — Direct Redshift Pull (`psycopg2`)

> This is the **alternative** data-access path. The main module path is **Path A** in `eda_example.ipynb`, which uses `data_io.load_data()` (boto3 `redshift-data` + `UNLOAD` to S3, IAM auth, no VPN). See its Setup section for the comparison.

**Use this notebook when**: you need a quick one-off sample on your laptop and you already have VPN + DB credentials. The output Parquet is what gets committed to `sample_data_from_redshift/` for offline development.

## Steps

1. **Connect to VPN** — `vpn.ao.zapsi.net`.
2. **Verify the connection** in the AWS Redshift Query Editor:
   <https://af-south-1.console.aws.amazon.com/sqlworkbench/home?region=af-south-1#/client>
3. **Provide the password via environment variable** — never hardcode it in the notebook.
   Create a `.env` file in this folder (already in `.gitignore`):
   ```
   REDSHIFT_PASSWORD=...
   ```
   Or export it in your shell before launching Jupyter:
   ```bash
   export REDSHIFT_PASSWORD='...'
   ```
4. **Install dependencies** (if not already installed via `requirements.txt`):
   ```bash
   pip install python-dotenv psycopg2-binary pyarrow
   ```
   Restart the kernel after installing.
5. **Run the cell below** — it connects, runs the sample query, and saves the result to `../sample_data_from_redshift/`.

⚠️ **Security**: the password is a real production credential. Read it from the environment, never paste it into the notebook. If it ever lands in a committed cell, **rotate it immediately** — git history retains it forever.

In [1]:
from dotenv import load_dotenv
import os
import psycopg2
import pandas as pd

# Loads the .env at the repo root (or any parent), pulling in `redshift_password`
load_dotenv()

# ====== CONFIG ======
REDSHIFT_HOST = "redshift-cluster-dsi.cl4o4mmtx9ir.af-south-1.redshift.amazonaws.com"
REDSHIFT_PORT = 5439
REDSHIFT_DB   = "prod"
REDSHIFT_USER = "awsuser"

# Read the password from the .env file — never hardcode it here.
REDSHIFT_PASSWORD = os.getenv("redshift_password")

if not REDSHIFT_PASSWORD:
    raise RuntimeError(
        "redshift_password is not set. Create a .env file at the repo root "
        "containing:\n    redshift_password=\"your-password\""
    )

# Example query — change to your table
QUERY = "SELECT * FROM prod.dth_churn_ml_training.training_features WHERE RANDOM() < 0.003;"

OUTPUT_PARQUET = "../sample_data_from_redshift/sample_from_prod.parquet"
# ====================

conn = psycopg2.connect(
    host=REDSHIFT_HOST,
    port=REDSHIFT_PORT,
    dbname=REDSHIFT_DB,
    user=REDSHIFT_USER,
    password=REDSHIFT_PASSWORD,
)

try:
    df = pd.read_sql(QUERY, conn)
    print(f"Fetched {len(df)} rows from Redshift")
    print(df.head())

    df.to_parquet(OUTPUT_PARQUET, index=False)
    print(f"Saved {len(df)} rows to {OUTPUT_PARQUET}")
finally:
    conn.close()

/var/folders/3d/dh5fxyvd55sbf5r8pv3bfclw0000gq/T/ipykernel_39743/2905626791.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(QUERY, conn)


Fetched 109676 rows from Redshift
   idconsumo  id_contaservico codigocontaservico  idconta iddim_date_inicio  \
0  499103375              381       100003710101      371        2025-04-05   
1  586039579           866748       100010080601   840886        2026-01-19   
2  590925884          2976212       100011840601  2919840        2026-02-03   
3  490774589          2614447       100015520301  2558459        2025-03-06   
4  491534294          3875952       100016781701  3828648        2025-03-09   

  iddim_date_fim  id_produto_actual tipo_produto_actual  tipo_subscricao  \
0     2025-04-07                 24              normal                7   
1     2026-01-22                 24            tafacil7                7   
2     2026-02-05                 24            tafacil7                7   
3     2025-04-04                 18              normal                7   
4     2025-03-11                 24            tafacil7                7   

  tipo_stb  ...  was_contacted  to

In [2]:
df

,idconsumo,id_contaservico,codigocontaservico,idconta,iddim_date_inicio,iddim_date_fim,id_produto_actual,tipo_produto_actual,tipo_subscricao,tipo_stb,...,was_contacted,topup_count,topup_total_value,topup_avg_value,topup_std_value,topup_cv_value,topup_days_since_last,used_selfcare,topup_type_nunique,topup_channel_nunique
0,499103375,381,100003710101,371,2025-04-05,2025-04-07,24,normal,7,HD,...,0,30,13359.65,445.32,668.152445,1.500387,2.0,0,4,1
1,586039579,866748,100010080601,840886,2026-01-19,2026-01-22,24,tafacil7,7,HD,...,0,29,28956.14,998.48,1826.613210,1.829394,3.0,0,7,0
2,590925884,2976212,100011840601,2919840,2026-02-03,2026-02-05,24,tafacil7,7,HD,...,0,5,3684.21,736.84,672.641641,0.912873,2.0,0,2,0
3,490774589,2614447,100015520301,2558459,2025-03-06,2025-04-04,18,normal,7,HD,...,0,7,48859.66,6979.95,7946.808904,1.138519,29.0,1,3,1
4,491534294,3875952,100016781701,3828648,2025-03-09,2025-03-11,24,tafacil7,7,HD,...,0,119,185026.24,1554.84,3463.966519,2.227860,2.0,1,7,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109671,586410775,4420949,240046780101,4380750,2026-01-20,2026-02-02,18,tafacil,7,HD,...,0,4,11578.96,2894.74,678.640600,0.234439,13.0,0,1,0
109672,521484289,4421503,240050360101,4381316,2025-06-23,2025-06-29,24,tafacil7,7,HD,...,0,8,8596.49,1074.56,434.188312,0.404061,6.0,0,3,1
109673,510404880,4421682,240051500101,4381500,2025-05-15,2025-05-18,24,normal,7,HD,...,0,16,11798.24,737.39,1153.901688,1.564846,3.0,0,3,1
109674,502491087,4421845,240052610101,4381668,2025-04-17,2025-04-22,18,tafacil7,7,HD,...,0,36,15368.42,426.90,620.765110,1.454123,5.0,0,4,1
